# Домашнее задание 8. Мониторинг

Прослеживаемость конвейера (и кода, и данных) должна закладываться **до проектирования** (в идеале, на основе синтезированного трафика от системы нагрузочного тестирования Locust/k6s/ab/...).


## Задание 5. Разработка схемы ML-системы для Virtual Product Placement

## Требования к системе
Система должна в реальном времени встраивать бренды (логотипы, рекламу) в видеопоток с учётом контекста сцены, геолокации пользователя и требований рекламодателя.

### Ключевые нефункциональные требования:
* **Задержка обработки кадра** < 200 мс (SLO)
* **Пропускная способность** до 10 000 RPS
* **Обнаружение дрейфа данных (PSI)** и деградации модели
* **Масштабируемость** и отказоустойчивость

## Выбор архитектуры: Kappa
В отличие от Lambda-архитектуры, мы отказываемся от раздельного пакетного (batch) уровня. Все данные обрабатываются через единый потоковый конвейер.

### Обоснование:
* **Единая кодовая база** для всей обработки (нет дублирования логики batch/stream).
* **Низкая задержка** – данные обрабатываются по мере поступления, нет ожидания накопления батча.
* **Возможность переобработки** – при необходимости можно «отмотать» смещение в Kafka и пересчитать метрики заново.

---

## Ключевые компоненты схемы


| Компонент | Назначение |
| :--- | :--- |
| **Источник видеопотока** (CDN / камера) | Генерирует кадры и контекст пользователя. |
| **Брокер сообщений** (Kafka) | Единый источник истины. Принимает сырые события (видеопоток, данные о пользователе) и хранит их в логах (топики `raw-frames`, `user-context`). |
| **Потоковый ML-сервис** (FastAPI + YOLO) | Считывает данные напрямую из Kafka. Выполняет детекцию объектов, определяет зоны для вставки, накладывает бренд (генеративный ИИ или шаблон). Попутно рассчитывает метрики (PSI, latency, precision). |
| **State store** (Redis / внутренний кэш) | Хранит временные сессии, статистику для расчёта PSI, контекст пользователя. Обеспечивает согласованность при распределённой обработке. |
| **Мониторинг** (Prometheus + Grafana) | Prometheus собирает метрики из ML-сервиса (latency, RPS, precision, IoU, PSI). Grafana визуализирует дашборды и отправляет алерты. |
| **Хранилище логов** (ClickHouse) | Принимает результаты инференса и исторические логи через Kafka Engine (или напрямую) для долгосрочной аналитики и бизнес-отчётов. |
| **Пайплайн переобучения** (CI/CD + SageMaker) | При срабатывании алерта PSI > 0.2 запускается дообучение модели на новых данных. Новая версия модели автоматически развёртывается в ML-сервисе. |

In [1]:
!apt-get update
!apt-get install graphviz -y
!pip install diagrams

Get:1 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:2 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:4 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:5 https://cli.github.com/packages stable InRelease [3,917 B]
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:7 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,644 kB]
Get:8 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:9 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Hit:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:11 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3,915 kB]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,295 kB]
Hit:13 https://ppa.launchpadcontent.net/graphi

In [3]:
from diagrams import Diagram, Cluster, Edge
from diagrams.aws.storage import S3
from diagrams.aws.compute import EC2
from diagrams.aws.ml import Sagemaker
from diagrams.onprem.queue import Kafka
from diagrams.onprem.database import Clickhouse
from diagrams.onprem.inmemory import Redis
from diagrams.onprem.monitoring import Prometheus, Grafana
from diagrams.programming.language import Python

with Diagram("Virtual Product Placement - Kappa Architecture", show=False, direction="LR"):
    video_source = EC2("Video Source\n(CDN/Camera)")
    kafka = Kafka("Kafka\n(raw-frames, user-context)")
    ml_service = Python("ML Service\n(FastAPI + YOLO)")
    state_store = Redis("State Store\n(sessions, PSI cache)")
    prom = Prometheus("Prometheus\n(metrics collection)")
    grafana = Grafana("Grafana\n(dashboards & alerts)")
    clickhouse = Clickhouse("ClickHouse\n(logs & analytics)")
    retrain = Sagemaker("Retrain Pipeline\n(trigger on PSI >0.2)")

    video_source >> Edge(label="video stream") >> kafka
    kafka >> Edge(label="consume frames") >> ml_service
    ml_service >> Edge(label="read/write state") >> state_store
    ml_service >> Edge(label="inference results") >> kafka
    ml_service >> Edge(label="expose /metrics") >> prom
    prom >> Edge(label="query") >> grafana
    kafka >> Edge(label="consume (Kafka Engine)") >> clickhouse
    prom >> Edge(label="PSI >0.2 alert", style="dashed") >> retrain
    retrain >> Edge(label="new model version", style="dashed") >> ml_service